# Day 3: Building the Complete Game

## Welcome to Day 3!

**What we've learned so far:**
- Day 1: Python basics, lists, board representation
- Day 2: Move logic, loops, captures, validation

**Today we'll learn:**
- Game loop (turn-based structure)
- Win conditions (how to end the game)
- Getting player input from keyboard
- Score tracking and announcements
- Making the game feel polished

By the end of today, you'll have a fully playable Mancala game!

---

## Part 1: Review - Complete Move Functions

Let's bring in all our code from Days 1 and 2:

In [ ]:
# ========================================
# COMPLETE GAME CODE FROM DAYS 1-2
# ========================================

def create_board():
    """Create a new Mancala board with starting positions."""
    return [4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0]


def display_board(board):
    """Display the Mancala board in a readable format."""
    print("\n" + "=" * 50)
    print("                MANCALA BOARD")
    print("=" * 50)
    print(f"Player 2 Store: [{board[13]:2d}]")
    print(f"Player 2:  {board[12]:2d}  {board[11]:2d}  {board[10]:2d}  {board[9]:2d}  {board[8]:2d}  {board[7]:2d}")
    print("           " + "―" * 25)
    print(f"Player 1:  {board[0]:2d}  {board[1]:2d}  {board[2]:2d}  {board[3]:2d}  {board[4]:2d}  {board[5]:2d}")
    print(f"Player 1 Store: [{board[6]:2d}]")
    print("           0   1   2   3   4   5   (pocket numbers)")
    print("=" * 50 + "\n")


def get_opposite_pocket(pocket):
    """Get the pocket directly opposite on the board."""
    return 12 - pocket


def is_valid_move(board, pocket, player):
    """Check if a move is valid."""
    if pocket < 0 or pocket > 12:
        return False, "Invalid pocket number."
    
    if player == 1:
        if pocket < 0 or pocket > 5:
            return False, "That pocket is not on your side. Choose 0-5."
    else:
        if pocket < 7 or pocket > 12:
            return False, "That pocket is not on your side. Choose 7-12."
    
    if board[pocket] == 0:
        return False, "That pocket is empty. Choose a different pocket."
    
    return True, "Valid move!"


def check_free_turn(last_position, player):
    """Check if player gets a free turn."""
    if player == 1 and last_position == 6:
        return True
    elif player == 2 and last_position == 13:
        return True
    return False


def check_capture(board, last_position, player):
    """Check if a capture occurred and execute it."""
    player_pockets = range(0, 6) if player == 1 else range(7, 13)
    player_store = 6 if player == 1 else 13
    
    if last_position not in player_pockets:
        return board
    
    if board[last_position] != 1:
        return board
    
    opposite = get_opposite_pocket(last_position)
    
    if board[opposite] == 0:
        return board
    
    captured_seeds = board[opposite] + board[last_position]
    print(f"\n💰 CAPTURE! Player {player} captures {captured_seeds} seeds!")
    
    board[player_store] += captured_seeds
    board[opposite] = 0
    board[last_position] = 0
    
    return board


def make_move(board, pocket, player, show_details=True):
    """Make a complete move with all Mancala rules."""
    valid, message = is_valid_move(board, pocket, player)
    if not valid:
        print(f"❌ Invalid move: {message}")
        return board, False
    
    if player == 1:
        opponent_store = 13
        player_store = 6
    else:
        opponent_store = 6
        player_store = 13
    
    seeds_in_hand = board[pocket]
    board[pocket] = 0
    
    if show_details:
        print(f"\nPlayer {player} picked up {seeds_in_hand} seeds from pocket {pocket}")
    
    current_position = pocket
    
    while seeds_in_hand > 0:
        current_position = (current_position + 1) % 14
        
        if current_position == opponent_store:
            continue
        
        board[current_position] += 1
        seeds_in_hand -= 1
    
    if show_details:
        print(f"Last seed landed at position {current_position}")
    
    board = check_capture(board, current_position, player)
    gets_free_turn = check_free_turn(current_position, player)
    
    if gets_free_turn and show_details:
        print(f"\n🎉 Player {player} gets another turn!")
    
    return board, gets_free_turn


print("✓ All functions loaded successfully!")

## Part 2: Win Conditions - When Does the Game End?

**Mancala ends when:**
One player's side (all 6 pockets) is completely empty.

**What happens then:**
- Remaining seeds on the other side go to that player's store
- Player with most seeds in their store wins!

Let's write functions to check this:

In [ ]:
def is_side_empty(board, player):
    """
    Check if a player's side is completely empty.
    
    Parameters:
    board (list): Current game board
    player (int): Which player to check (1 or 2)
    
    Returns:
    bool: True if all pockets on player's side are empty
    """
    if player == 1:
        # Check pockets 0-5
        return all(board[i] == 0 for i in range(0, 6))
    else:
        # Check pockets 7-12
        return all(board[i] == 0 for i in range(7, 13))


# Test the function
test_board_1 = [0, 0, 0, 0, 0, 0, 20, 4, 4, 4, 4, 4, 4, 8]
print("Test board 1:")
display_board(test_board_1)
print("Player 1's side empty?", is_side_empty(test_board_1, 1))
print("Player 2's side empty?", is_side_empty(test_board_1, 2))

test_board_2 = [2, 0, 3, 0, 1, 0, 15, 0, 0, 0, 0, 0, 0, 12]
print("\nTest board 2:")
display_board(test_board_2)
print("Player 1's side empty?", is_side_empty(test_board_2, 1))
print("Player 2's side empty?", is_side_empty(test_board_2, 2))

### Understanding `all()`

The `all()` function checks if ALL items in a sequence are True:

In [ ]:
# Examples of all()
print("all([True, True, True]):", all([True, True, True]))
print("all([True, False, True]):", all([True, False, True]))
print("all([1 > 0, 2 > 0, 3 > 0]):", all([1 > 0, 2 > 0, 3 > 0]))

# Checking if all numbers are zero
numbers = [0, 0, 0, 0]
print("\nAll zeros?", all(n == 0 for n in numbers))

numbers2 = [0, 1, 0, 0]
print("All zeros?", all(n == 0 for n in numbers2))

## Part 3: Collecting Remaining Seeds

When the game ends, we need to collect all remaining seeds:

In [ ]:
def collect_remaining_seeds(board):
    """
    Collect all remaining seeds to their respective stores.
    Called when game ends.
    
    Parameters:
    board (list): Current game board
    
    Returns:
    list: Updated board with all seeds collected
    """
    # Collect Player 1's remaining seeds (pockets 0-5)
    player1_remaining = sum(board[i] for i in range(0, 6))
    board[6] += player1_remaining  # Add to Player 1's store
    
    # Empty Player 1's pockets
    for i in range(0, 6):
        board[i] = 0
    
    # Collect Player 2's remaining seeds (pockets 7-12)
    player2_remaining = sum(board[i] for i in range(7, 13))
    board[13] += player2_remaining  # Add to Player 2's store
    
    # Empty Player 2's pockets
    for i in range(7, 13):
        board[i] = 0
    
    if player1_remaining > 0:
        print(f"\nPlayer 1 collects {player1_remaining} remaining seeds")
    if player2_remaining > 0:
        print(f"Player 2 collects {player2_remaining} remaining seeds")
    
    return board


# Test collecting seeds
test_end_board = [0, 0, 0, 0, 0, 0, 18, 3, 2, 4, 1, 5, 2, 13]
print("Before collecting:")
display_board(test_end_board)

test_end_board = collect_remaining_seeds(test_end_board)
print("\nAfter collecting:")
display_board(test_end_board)

## Part 4: Determining the Winner

Once all seeds are collected, who won?

In [ ]:
def get_winner(board):
    """
    Determine the winner based on store counts.
    
    Parameters:
    board (list): Final game board
    
    Returns:
    int: Winner (1, 2, or 0 for tie)
    """
    player1_score = board[6]
    player2_score = board[13]
    
    print(f"\nFinal Scores:")
    print(f"Player 1: {player1_score} seeds")
    print(f"Player 2: {player2_score} seeds")
    
    if player1_score > player2_score:
        return 1
    elif player2_score > player1_score:
        return 2
    else:
        return 0  # Tie


def announce_winner(winner):
    """
    Display winner announcement.
    
    Parameters:
    winner (int): 1, 2, or 0 for tie
    """
    print("\n" + "=" * 50)
    print("                 GAME OVER!")
    print("=" * 50)
    
    if winner == 0:
        print("\n🤝 It's a TIE! Both players are equally skilled!")
    else:
        print(f"\n🏆 PLAYER {winner} WINS! Congratulations! 🏆")
    
    print("\n" + "=" * 50 + "\n")


# Test winner functions
final_board = [0, 0, 0, 0, 0, 0, 28, 0, 0, 0, 0, 0, 0, 20]
display_board(final_board)
winner = get_winner(final_board)
announce_winner(winner)

## Part 5: Getting Player Input

Now we need to get moves from the player. We'll use the `input()` function:

In [ ]:
# Basic input example
name = input("What is your name? ")
print(f"Hello, {name}!")

### Converting Input to Numbers

`input()` always returns a string, but we need a number for pocket selection:

In [ ]:
# Getting a number from user
age_string = input("How old are you? ")
print(f"You entered: '{age_string}' (type: {type(age_string)})")

age_number = int(age_string)  # Convert to integer
print(f"Converted to: {age_number} (type: {type(age_number)})")
print(f"Next year you'll be {age_number + 1}")

### Handling Invalid Input

What if the player types something that's not a number? We need error handling:

In [ ]:
def get_pocket_choice(player):
    """
    Get a valid pocket choice from the player.
    Keeps asking until they enter a valid number.
    
    Parameters:
    player (int): Current player (1 or 2)
    
    Returns:
    int: The chosen pocket number
    """
    if player == 1:
        prompt = "Player 1, choose a pocket (0-5): "
    else:
        prompt = "Player 2, choose a pocket (7-12): "
    
    while True:  # Keep asking until valid input
        try:
            choice = input(prompt)
            pocket = int(choice)  # Try to convert to number
            return pocket  # If successful, return it
        except ValueError:
            # This runs if int() fails (non-number entered)
            print("❌ Please enter a number!")


# Test it (try entering invalid input like "abc" or "2.5")
# choice = get_pocket_choice(1)
# print(f"You chose pocket {choice}")

### Understanding Try-Except

`try-except` lets us handle errors gracefully:

In [ ]:
# Example 1: This would crash without try-except
user_input = "hello"

try:
    number = int(user_input)
    print(f"You entered: {number}")
except ValueError:
    print("That's not a number!")

# Example 2: Division by zero
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Can't divide by zero!")

## Part 6: The Game Loop Structure

A **game loop** repeats these steps:
1. Display game state
2. Get player input
3. Make move
4. Check if game is over
5. Switch turns (if no free turn)
6. Repeat

Here's the basic structure:

In [ ]:
def game_loop():
    """
    Main game loop - runs the entire game.
    """
    # Initialize game
    board = create_board()
    current_player = 1
    move_count = 0
    
    print("\n" + "=" * 50)
    print("           WELCOME TO MANCALA!")
    print("=" * 50)
    print("\nPlayer 1: Use pockets 0-5")
    print("Player 2: Use pockets 7-12")
    print("\nLand your last seed in your store for a free turn!")
    print("Land in an empty pocket to capture opponent's seeds!\n")
    
    # Main game loop
    while True:
        # Check if game is over
        if is_side_empty(board, 1) or is_side_empty(board, 2):
            print("\n🏁 One side is empty - Game Over!")
            board = collect_remaining_seeds(board)
            display_board(board)
            winner = get_winner(board)
            announce_winner(winner)
            break  # Exit the loop
        
        # Display current state
        display_board(board)
        print(f"\n--- Move #{move_count + 1} ---")
        print(f"Current turn: Player {current_player}")
        
        # Get player's move
        pocket = get_pocket_choice(current_player)
        
        # Make the move
        board, gets_free_turn = make_move(board, pocket, current_player)
        move_count += 1
        
        # Switch players (unless free turn)
        if not gets_free_turn:
            current_player = 2 if current_player == 1 else 1
        
        # Small delay for readability
        input("\nPress Enter to continue...")
    
    print(f"\nTotal moves played: {move_count}")
    print("Thanks for playing Mancala!\n")


# To start the game, run:
# game_loop()

## Part 7: Adding Game Statistics

Let's track some interesting stats during the game:

In [ ]:
def game_loop_with_stats():
    """
    Enhanced game loop with statistics tracking.
    """
    # Initialize game
    board = create_board()
    current_player = 1
    
    # Statistics
    move_count = 0
    player1_moves = 0
    player2_moves = 0
    free_turns = 0
    captures = 0
    
    print("\n" + "=" * 50)
    print("           WELCOME TO MANCALA!")
    print("=" * 50)
    print("\nPlayer 1: Use pockets 0-5")
    print("Player 2: Use pockets 7-12")
    print("\nLand your last seed in your store for a free turn!")
    print("Land in an empty pocket to capture opponent's seeds!\n")
    
    # Main game loop
    while True:
        # Check if game is over
        if is_side_empty(board, 1) or is_side_empty(board, 2):
            print("\n🏁 One side is empty - Game Over!")
            board = collect_remaining_seeds(board)
            display_board(board)
            winner = get_winner(board)
            announce_winner(winner)
            
            # Display statistics
            print("\n📊 GAME STATISTICS 📊")
            print("=" * 50)
            print(f"Total moves: {move_count}")
            print(f"Player 1 moves: {player1_moves}")
            print(f"Player 2 moves: {player2_moves}")
            print(f"Free turns earned: {free_turns}")
            print(f"Captures made: {captures}")
            print("=" * 50 + "\n")
            break
        
        # Display current state
        display_board(board)
        print(f"\n--- Move #{move_count + 1} ---")
        print(f"Current turn: Player {current_player}")
        
        # Get player's move
        pocket = get_pocket_choice(current_player)
        
        # Track captures before move
        store_before = board[6] + board[13]
        
        # Make the move
        board, gets_free_turn = make_move(board, pocket, current_player)
        
        # Track captures after move
        store_after = board[6] + board[13]
        if store_after > store_before + 4:  # More than just distributed seeds
            captures += 1
        
        # Update statistics
        move_count += 1
        if current_player == 1:
            player1_moves += 1
        else:
            player2_moves += 1
        
        if gets_free_turn:
            free_turns += 1
        
        # Switch players (unless free turn)
        if not gets_free_turn:
            current_player = 2 if current_player == 1 else 1
        
        # Small delay for readability
        input("\nPress Enter to continue...")
    
    print("Thanks for playing Mancala!\n")


# To start the enhanced game, run:
# game_loop_with_stats()

## Part 8: Adding Player Names

Let's make it more personal by using player names:

In [ ]:
def display_board_with_names(board, player1_name, player2_name):
    """
    Display the Mancala board with player names.
    """
    print("\n" + "=" * 50)
    print("                MANCALA BOARD")
    print("=" * 50)
    print(f"{player2_name}'s Store: [{board[13]:2d}]")
    print(f"{player2_name}:  {board[12]:2d}  {board[11]:2d}  {board[10]:2d}  {board[9]:2d}  {board[8]:2d}  {board[7]:2d}")
    print("           " + "―" * 25)
    print(f"{player1_name}:  {board[0]:2d}  {board[1]:2d}  {board[2]:2d}  {board[3]:2d}  {board[4]:2d}  {board[5]:2d}")
    print(f"{player1_name}'s Store: [{board[6]:2d}]")
    print("           0   1   2   3   4   5   (pocket numbers)")
    print("=" * 50 + "\n")


# Test it
test_board = create_board()
display_board_with_names(test_board, "Alice", "Bob")

## Part 9: Complete Game with All Features

Here's the final, polished version with all our improvements:

In [ ]:
def play_mancala():
    """
    Complete Mancala game with all features!
    """
    # Get player names
    print("\n" + "=" * 50)
    print("           WELCOME TO MANCALA!")
    print("=" * 50 + "\n")
    
    player1_name = input("Player 1, enter your name: ") or "Player 1"
    player2_name = input("Player 2, enter your name: ") or "Player 2"
    
    print(f"\nWelcome {player1_name} and {player2_name}!")
    print(f"\n{player1_name}: Use pockets 0-5")
    print(f"{player2_name}: Use pockets 7-12")
    print("\nLand your last seed in your store for a free turn!")
    print("Land in an empty pocket to capture opponent's seeds!\n")
    
    input("Press Enter to start the game...")
    
    # Initialize game
    board = create_board()
    current_player = 1
    player_names = {1: player1_name, 2: player2_name}
    
    # Statistics
    stats = {
        'moves': 0,
        'player1_moves': 0,
        'player2_moves': 0,
        'free_turns': 0,
        'captures': 0
    }
    
    # Main game loop
    while True:
        # Check if game is over
        if is_side_empty(board, 1) or is_side_empty(board, 2):
            print("\n🏁 One side is empty - Game Over!")
            board = collect_remaining_seeds(board)
            display_board_with_names(board, player1_name, player2_name)
            
            # Determine winner
            player1_score = board[6]
            player2_score = board[13]
            
            print(f"\nFinal Scores:")
            print(f"{player1_name}: {player1_score} seeds")
            print(f"{player2_name}: {player2_score} seeds")
            
            print("\n" + "=" * 50)
            print("                 GAME OVER!")
            print("=" * 50)
            
            if player1_score > player2_score:
                print(f"\n🏆 {player1_name} WINS! Congratulations! 🏆")
            elif player2_score > player1_score:
                print(f"\n🏆 {player2_name} WINS! Congratulations! 🏆")
            else:
                print("\n🤝 It's a TIE! Both players are equally skilled!")
            
            print("\n" + "=" * 50)
            
            # Display statistics
            print("\n📊 GAME STATISTICS 📊")
            print("=" * 50)
            print(f"Total moves: {stats['moves']}")
            print(f"{player1_name} moves: {stats['player1_moves']}")
            print(f"{player2_name} moves: {stats['player2_moves']}")
            print(f"Free turns earned: {stats['free_turns']}")
            print(f"Captures made: {stats['captures']}")
            print("=" * 50 + "\n")
            break
        
        # Display current state
        display_board_with_names(board, player1_name, player2_name)
        print(f"\n--- Move #{stats['moves'] + 1} ---")
        print(f"Current turn: {player_names[current_player]}")
        
        # Get player's move
        pocket = get_pocket_choice(current_player)
        
        # Track state before move
        store_before = board[6] + board[13]
        
        # Make the move
        board, gets_free_turn = make_move(board, pocket, current_player)
        
        # Track captures
        store_after = board[6] + board[13]
        if store_after > store_before + 4:
            stats['captures'] += 1
        
        # Update statistics
        stats['moves'] += 1
        if current_player == 1:
            stats['player1_moves'] += 1
        else:
            stats['player2_moves'] += 1
        
        if gets_free_turn:
            stats['free_turns'] += 1
        
        # Switch players (unless free turn)
        if not gets_free_turn:
            current_player = 2 if current_player == 1 else 1
        
        # Small delay for readability
        input("\nPress Enter to continue...")
    
    print("Thanks for playing Mancala!\n")


# To play the complete game, run:
# play_mancala()

## Part 10: Saving Your Game to a File

Let's save the complete game as a Python file you can run outside of Jupyter:

In [ ]:
# This will create a standalone Python file
game_code = '''
#!/usr/bin/env python3
"""
MANCALA GAME
A complete implementation of the classic two-player strategy game.
Created during Game Development with Python Workshop
"""

def create_board():
    """Create a new Mancala board with starting positions."""
    return [4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0]


def display_board_with_names(board, player1_name, player2_name):
    """Display the Mancala board with player names."""
    print("\\n" + "=" * 50)
    print("                MANCALA BOARD")
    print("=" * 50)
    print(f"{player2_name}\'s Store: [{board[13]:2d}]")
    print(f"{player2_name}:  {board[12]:2d}  {board[11]:2d}  {board[10]:2d}  {board[9]:2d}  {board[8]:2d}  {board[7]:2d}")
    print("           " + "―" * 25)
    print(f"{player1_name}:  {board[0]:2d}  {board[1]:2d}  {board[2]:2d}  {board[3]:2d}  {board[4]:2d}  {board[5]:2d}")
    print(f"{player1_name}\'s Store: [{board[6]:2d}]")
    print("           0   1   2   3   4   5   (pocket numbers)")
    print("=" * 50 + "\\n")


# [Include all other functions here...]

if __name__ == "__main__":
    play_mancala()
'''

# Save to file
with open('mancala_game.py', 'w') as f:
    f.write(game_code)

print("✓ Game saved to 'mancala_game.py'")
print("Run it with: python mancala_game.py")

## Day 3 Summary - What We Built Today

✅ **Win Conditions:** Detecting when game ends (side empty)

✅ **Seed Collection:** Gathering remaining seeds at game end

✅ **Winner Determination:** Comparing scores and announcing results

✅ **Player Input:** Getting moves from keyboard with validation

✅ **Error Handling:** Using try-except for robust input

✅ **Game Loop:** Turn-based structure that runs the entire game

✅ **Statistics Tracking:** Counting moves, captures, free turns

✅ **Personalization:** Using player names throughout

✅ **Complete Game:** Fully playable from start to finish!

---

## Tomorrow's Preview - Day 4

On our final day, we'll:
- Add advanced features (undo move, hints, difficulty levels)
- Create an AI opponent (basic strategy)
- Polish the user interface
- Document our code professionally
- Create a developer's guide
- Prepare for the showcase!

---

## Practice Challenges

1. **Add a move timer:** Track how long each player takes
2. **Save game history:** Store all moves in a list
3. **Replay mode:** Play back the game move-by-move
4. **Color coding:** Use ANSI colors to highlight current player
5. **Sound effects:** Play beeps for captures and free turns (using `print('\a')`)

---

## Key Vocabulary

- **Game Loop:** Main loop that runs the game until completion
- **Win Condition:** Rule that determines when game ends
- **Input Validation:** Checking if user input is acceptable
- **Try-Except:** Error handling structure
- **Break:** Exit a loop immediately
- **All():** Function that checks if all items are True
- **Sum():** Function that adds up all numbers in a sequence
- **Statistics:** Data collected about game performance